In [46]:
import pandas as pd

deltas = pd.read_csv("../ranking_methods/data/total_deltas.csv")
limes = pd.read_csv("../ranking_methods/data/lime_by_hadm.csv")

In [47]:
deltas = deltas[["subject_id", "hadm_id", "feature", "delta_mean", "delta_mean_abs", "direction_mode", "direction_consistency"]]

In [48]:
limes = limes[["subject_id", "hadm_id", "feature", "lime_mean", "lime_mean_abs", "value_mean"]]

In [49]:
shap = pd.read_csv("../ranking_methods/data/shap_matrix.csv")
shap_abs = pd.read_csv("../ranking_methods/data/shap_matrix_abs.csv")
shap_long = shap.melt(id_vars=['hadm_id'], var_name='feature', value_name='shap_mean')
shap_abs_long = shap_abs.melt(id_vars=['hadm_id'], var_name='feature', value_name='shap_mean_abs')
shap_combined = shap_long.merge(shap_abs_long, on=['hadm_id', 'feature'], how='inner')

features_to_analyze = deltas['feature'].unique()
shap_combined = shap_combined[shap_combined['feature'].isin(features_to_analyze)]

In [50]:
shap_bck = pd.read_csv("../ranking_methods/data/shap_matrix_background.csv")
shap_abs_bck = pd.read_csv("../ranking_methods/data/shap_matrix_abs_background.csv")
shap_long_bck = shap_bck.melt(id_vars=['hadm_id'], var_name='feature', value_name='shap_mean')
shap_abs_long_bck = shap_abs_bck.melt(id_vars=['hadm_id'], var_name='feature', value_name='shap_mean_abs')
shap_combined_bck = shap_long_bck.merge(shap_abs_long_bck, on=['hadm_id', 'feature'], how='inner')

shap_combined_bck = shap_combined_bck[shap_combined_bck['feature'].isin(features_to_analyze)]

In [51]:
lab_names = {
    '50983': 'Sodium',
    '50971': 'Potassium',
    '50902': 'Chloride',
    '50882': 'Bicarbonate',
    '50912': 'Creatinine',
    '51006': 'BUN',
    '50931': 'Glucose',
    '50893': 'Calcium',
    '50868': 'Anion Gap',
    '51222': 'Hemoglobin',
    '51301': 'WBC',
    '51265': 'Platelet Count',
    '51221': 'Hematocrit',
    '51250': 'MCV',
    '51277': 'RDW',
    '50960': 'Magnesium',
    '50970': 'Phosphate',
    '51248': 'MCH',
    '51249': 'MCHC',
    '51279': 'RBC'
}

icd_names = {
    'I10': 'Essential (primary) hypertension',
    'E785': 'Hyperlipidemia, unspecified',
    'K219': 'Gastroesophageal reflux disease without esophagitis',
    'Z87891': 'Personal history of nicotine dependence',
    'I2510': 'Atherosclerotic heart disease of native coronary artery without angina pectoris',
    'N179': 'Acute kidney failure, unspecified',
    'F329': 'Major depressive disorder, single episode, unspecified',
    'I4891': 'Unspecified atrial fibrillation',
    'Z7901': 'Long term (current) use of anticoagulants',
    'F419': 'Anxiety disorder, unspecified',
    'E119': 'Type 2 diabetes mellitus without complications',
    'E039': 'Hypothyroidism, unspecified',
    'Z794': 'Long term (current) use of insulin',
    'D649': 'Anemia, unspecified',
    'N390': 'Urinary tract infection, site not specified'
}

ccsr_names = {
    'FAC021': 'Personal/family history of disease',
    'FAC025': 'Other specified status',
    'END011': 'Fluid and electrolyte disorders',
    'CIR011': 'Coronary atherosclerosis and other heart disease',
    'END010': 'Disorders of lipid metabolism',
    'CIR007': 'Essential hypertension',
    'END003': 'Diabetes mellitus with complication',
    'CIR019': 'Heart failure',
    'DIG004': 'Esophageal disorders',
    'CIR017': 'Cardiac dysrhythmias',
    'CIR008': 'Hypertension with complications and secondary hypertension',
    'BLD003': 'Aplastic anemia',
    'EXT027': 'External cause codes: place of occurrence of the external cause',
    'GEN002': 'Acute and unspecified renal failure',
    'END009': 'Obesity'
}

def map_feature_name(feature_name):
    if feature_name.startswith('lab_') and feature_name.endswith('_daily'):
        code = feature_name.replace('lab_', '').replace('_daily', '')
        if code in lab_names:
            return f"{lab_names[code]} ({code})"
    elif feature_name.startswith('icd_'):
        code = feature_name.replace('icd_', '')
        if code in icd_names:
            return f"{icd_names[code]} ({code})"
    elif feature_name.startswith('ccsr_'):
        code = feature_name.replace('ccsr_', '')
        if code in ccsr_names:
            return f"{ccsr_names[code]} ({code})"
    return feature_name

In [52]:
limes['feature'] = limes['feature'].apply(map_feature_name)
deltas['feature'] = deltas['feature'].apply(map_feature_name)
shap_combined_bck['feature'] = shap_combined_bck['feature'].apply(map_feature_name)
shap_combined['feature'] = shap_combined['feature'].apply(map_feature_name)

### Test examples (1440)

In [19]:
import numpy as np

test_df = pd.read_parquet('../training_ml/artifacts/test_data_with_predictions.parquet')
hospital_df = (
    test_df
    .sort_values(["subject_id", "hadm_id", "day"])
    .groupby(["subject_id", "hadm_id"])
    .last()
    .reset_index()
)

threshold = 0.2158
hospital_df["risk_scaled"] = np.where(
    hospital_df["predicted_proba"] <= threshold,
    0.5 * hospital_df["predicted_proba"] / threshold,
    0.5 + 0.5 * (
        (hospital_df["predicted_proba"] - threshold)
        / (hospital_df["predicted_proba"].max() - threshold)
    )
)

def prediction_group(row):
    if row.true_class == 1 and row.predicted_class == 1:
        return "TP"
    elif row.true_class == 0 and row.predicted_class == 0:
        return "TN"
    elif row.true_class == 0 and row.predicted_class == 1:
        return "FP"
    else:
        return "FN"

hospital_df["group"] = hospital_df.apply(prediction_group, axis=1)
icd_cols = [c for c in hospital_df.columns if c.startswith("icd_")]
ccsr_cols = [c for c in hospital_df.columns if c.startswith("ccsr_")]
lab_cols = [c for c in hospital_df.columns if c.startswith("lab_")]

hospital_df["has_icd"] = hospital_df[icd_cols].sum(axis=1) > 0
hospital_df["has_ccsr"] = hospital_df[ccsr_cols].sum(axis=1) > 0
hospital_df["has_labs"] = hospital_df[lab_cols].notna().any(axis=1)
hospital_df["context_type"] = np.select(
    [
        hospital_df["has_icd"] & hospital_df["has_ccsr"] & hospital_df["has_labs"],

        (hospital_df["has_icd"] | hospital_df["has_ccsr"]) &
        (~hospital_df["has_labs"]),

        (~hospital_df["has_icd"]) &
        (~hospital_df["has_ccsr"]) &
        hospital_df["has_labs"],

        (~hospital_df["has_icd"]) &
        (~hospital_df["has_ccsr"]) &
        (~hospital_df["has_labs"]),
    ],
    [
        "full",
        "diagnoses_only",
        "labs_only",
        "empty",
    ],
    default="other"
)

In [20]:
selected = []

contexts = [
    "full",
    "diagnoses_only",
    "labs_only",
    "empty"
]

for group in ["TP", "TN", "FP", "FN"]:
    df_group = hospital_df[hospital_df["group"] == group]
    for context in contexts:
        df = df_group[df_group["context_type"] == context]
        if len(df) == 0:
            continue
        low = df.nsmallest(30, "predicted_proba")
        high = df.nlargest(30, "predicted_proba")
        mid = (
            df.iloc[
                (
                    df["predicted_proba"]
                    - df["predicted_proba"].median()
                ).abs().argsort()
            ]
            .head(30)
        )

        selected.extend([low, mid, high])

sample = (
    pd.concat(selected)
      .drop_duplicates(subset="hadm_id")
      .reset_index(drop=True)
)

### EHR formatting

In [53]:
def get_patient_full_data(subject_id, hadm_id, total_data, risk_score=False):
    patient_raw = total_data[
        (total_data['subject_id'] == subject_id) &
        (total_data['hadm_id'] == hadm_id)
    ].sort_values("day")

    patient = patient_raw.iloc[-1].to_dict()

    json_context = {
        "demographics": {},
        "diagnoses": {},
        "laboratory_values": {},
        "clinical_indicators": {}
    }
    if risk_score:
        json_context = {
            #"risk_score": {},
            "readmission": {},
            "demographics": {},
            "diagnoses": {},
            "laboratory_values": {},
            "clinical_indicators": {}
        }
        
        #json_context["risk_score"] = round(patient.get("risk_scaled", None), 2)
        json_context["readmission"] = "True" if patient.get("true_class", None) == 1 else "False"

    json_context["demographics"]["gender"] = (
        "Male" if patient["gender_male"] == 1 else "Female"
    )
    json_context["demographics"]["age"] = (patient["age"])
    json_context["demographics"]["length_of_stay"] = (patient["los"])

    race_cols = [c for c in total_data.columns if c.startswith("race_")]

    for col in race_cols:
        if patient.get(col) == 1:
            json_context["demographics"]["race"] = (
                col.replace("race_", "")
                .replace("_", " ")
                .title()
            )
            break

    insurance_cols = [col for col in total_data.columns if col.startswith('insurance_')]
    for col in insurance_cols:
        if col in patient and patient[col] == 1:
            json_context["demographics"]["insurance"] = col.replace('insurance_', '')
            break
    if 'insurance' not in json_context["demographics"]:
        json_context["demographics"]["insurance"] = 'Unknown'

    admission_type_cols = [col for col in total_data.columns if col.startswith('admission_type_')]
    for col in admission_type_cols:
        if col in patient and patient[col] == 1:
            json_context["demographics"]["admission_type"] = col.replace('admission_type_', '').replace('_', ' ').title()
            break
    if 'admission_type' not in json_context["demographics"]:
        json_context["demographics"]["admission_type"] = 'Unknown'

    discharge_cols = [col for col in total_data.columns if col.startswith('discharge_location_')]
    for col in discharge_cols:
        if col in patient and patient[col] == 1:
            json_context["demographics"]["discharge_location"] = col.replace('discharge_location_', '').replace('_', ' ').title()
            break
    if 'discharge_location' not in json_context["demographics"]:
        json_context["demographics"]["discharge_location"] = 'Unknown'

    adm_location_cols = [col for col in total_data.columns if col.startswith('admission_location_')]
    for col in adm_location_cols:
        if col in patient and patient[col] == 1:
            json_context["demographics"]["admission_location"] = col.replace('admission_location_', '').replace('_', ' ').title()
            break
    if 'admission_location' not in json_context["demographics"]:
        json_context["demographics"]["admission_location"] = 'Unknown'

    icd_list = []
    icd_cols = [c for c in total_data.columns if c.startswith("icd_")]
    for col in icd_cols:
        if patient.get(col) == 1:
            icd_list.append(map_feature_name(col))
    json_context["diagnoses"]["icd"] = icd_list

    ccsr_list = []
    ccsr_cols = [c for c in total_data.columns if c.startswith("ccsr_")]
    for col in ccsr_cols:
        if patient.get(col) == 1:
            ccsr_list.append(map_feature_name(col))
    json_context["diagnoses"]["ccsr"] = ccsr_list

    lab_cols = [
        c for c in total_data.columns
        if c.startswith("lab_") and c.endswith("_daily")]
    for col in lab_cols:
        value = patient.get(col)
        if pd.notna(value):
            json_context["laboratory_values"][map_feature_name(col)] = round(float(value), 2)

    other_features = [
        "num_diagnoses",
        "num_chronic",
        "comorbidity_score",
        "num_medications_daily",
        "prior_admissions_12mo",
        "cumulative_procedures",
        "cumulative_medications",
        "num_procedures_daily"
    ]

    for feat in other_features:
        value = patient.get(feat)
        if pd.notna(value):
            json_context["clinical_indicators"][feat] = float(value)

    return {
        "subject_id": int(subject_id),
        "hadm_id": int(hadm_id),
        "json_context": json_context
    }

In [54]:
patients_json = []

for _, row in sample.iterrows():
    patient = get_patient_full_data(
        row.subject_id,
        row.hadm_id,
        sample,
        risk_score=True
    )
    patients_json.append(patient)

In [55]:
patients_json

[{'subject_id': 14849966,
  'hadm_id': 26616701,
  'json_context': {'readmission': 'True',
   'demographics': {'gender': 'Female',
    'age': 78,
    'length_of_stay': 19,
    'race': 'White - Other European',
    'insurance': 'Medicare',
    'admission_type': 'Ew Emer.',
    'discharge_location': 'Skilled Nursing Facility',
    'admission_location': 'Transfer From Hospital'},
   'diagnoses': {'icd': ['Hyperlipidemia, unspecified (E785)',
     'Gastroesophageal reflux disease without esophagitis (K219)',
     'Atherosclerotic heart disease of native coronary artery without angina pectoris (I2510)',
     'Acute kidney failure, unspecified (N179)',
     'Unspecified atrial fibrillation (I4891)'],
    'ccsr': ['Personal/family history of disease (FAC021)',
     'Other specified status (FAC025)',
     'Coronary atherosclerosis and other heart disease (CIR011)',
     'Disorders of lipid metabolism (END010)',
     'Diabetes mellitus with complication (END003)',
     'Esophageal disorders (DI

### Sorting top-10 features that exist in EHR

In [56]:
clinical_ranges = {
    'lab_50983_daily': {'low': 135, 'high': 145}, #sodium
    'lab_50971_daily': {'low': 3.5, 'high': 5}, #potassium
    'lab_50902_daily': {'low': 95, 'high': 105}, #chloride
    'lab_50882_daily': {'low': 22, 'high': 32}, #bicarbonate
    'lab_50912_daily': {'low': 0.8, 'high': 1.3}, #creatinine
    'lab_51006_daily': {'low': 8, 'high': 21}, #BUN
    'lab_50931_daily': {'low': 65, 'high': 110}, #glucose
    'lab_50893_daily': {'low': 8.5, 'high': 10.2}, #calcium
    'lab_50868_daily': {'low': 10, 'high': 18}, #anion gap
    'lab_51222_daily': {
        'male': {'low': 13, 'high': 17},
        'female': {'low': 12, 'high': 15}
    }, #hemoglobin
    'lab_51301_daily': {'low': 4, 'high': 10}, #WBC
    'lab_51265_daily': {'low': 150, 'high': 400}, #Platelet Count
    'lab_51221_daily': {
        'male': {'low': 40, 'high': 52},
        'female': {'low': 36, 'high': 47}
    }, #Hematocrit
    'lab_51250_daily': {'low': 80, 'high': 100}, #MCV
    'lab_51277_daily': {'low': 11.5, 'high': 14.5}, #RDW
    'lab_50960_daily': {'low': 1.5, 'high': 2}, #magnesium
    'lab_50970_daily': {'low': 2.7, 'high': 4.5}, #phosphate
    'lab_51248_daily': {'low': 26, 'high': 32}, #MCH
    'lab_51249_daily': {'low': 30, 'high': 35}, #MCHC
    'lab_51279_daily': {
        'male': {'low': 4.5, 'high': 5.9},
        'female': {'low': 4, 'high': 5.2}
    } #RBC
}

def correct_shap_signs(top_features, ehr_info):
    corrected_features = {}
    
    is_male = ehr_info.get("gender_male", 0)
    gender_key = "male" if (is_male == 1 or is_male is True) else "female"
    
    patient_labs = ehr_info.get('laboratory_values', {})
    clinical_indicators = ehr_info.get('clinical_indicators', {})
    demographics = ehr_info.get('demographics', {})

    for idx, row in top_features.iterrows():
        feature = row['feature']
        shap_val = row['shap_mean']
        corrected_val = shap_val
        
        always_increase_risk_features = [
            'Obesity (END009)', 
            'Heart failure (CIR019)', 
            'Hyperlipidemia, unspecified (E785)', 
            'Disorders of lipid metabolism (END010)',
            'Diabetes mellitus with complication (END003)'
        ]
        if any(x.lower() in feature.lower() for x in always_increase_risk_features):
            if shap_val < 0:
                corrected_val = abs(shap_val)

        elif 'medications' in feature.lower():
            actual_val = float(clinical_indicators.get(feature, 0))
            
            if actual_val == 0:
                if shap_val > 0:
                    corrected_val = -abs(shap_val)
            elif actual_val >= 5:
                if shap_val < 0:
                    corrected_val = abs(shap_val)

        elif 'procedures' in feature.lower():
            actual_val = float(clinical_indicators.get(feature, 0))
            
            if actual_val == 0:
                if shap_val > 0:
                    corrected_val = -abs(shap_val)
            elif actual_val > 5:
                if shap_val < 0:
                    corrected_val = abs(shap_val)

        elif 'age' in feature.lower():
            actual_val = float(demographics.get(feature, 0))
            if actual_val < 65:
                if shap_val > 0:
                    corrected_val = -abs(shap_val)
            elif actual_val >= 65:
                if shap_val < 0:
                    corrected_val = abs(shap_val)

        elif feature in ['comorbidity_score', 'num_chronic', 'num_diagnoses']:
            actual_indicator_str = clinical_indicators.get(feature)
            
            if actual_indicator_str is not None:
                actual_ind_val = float(actual_indicator_str)
                
                thresholds = {
                    'comorbidity_score': 2.0,
                    'num_chronic': 2.0,
                    'num_diagnoses': 7.0
                }
                
                max_safe_value = thresholds.get(feature, 0)
                
                if actual_ind_val <= max_safe_value:
                    if shap_val > 0:
                        corrected_val = -abs(shap_val)
                        
                else:
                    if shap_val < 0:
                        corrected_val = abs(shap_val)

        else:
            import re
            lab_id_match = re.search(r'\((\d{5})\)', feature)
            
            if lab_id_match:
                lab_id = lab_id_match.group(1)
                range_key = f"lab_{lab_id}_daily"
                
                if range_key in clinical_ranges:
                    actual_lab_str = patient_labs.get(feature)
                    
                    if actual_lab_str is not None:
                        actual_lab_val = float(actual_lab_str)
                        norm_data = clinical_ranges[range_key]
                        
                        if 'male' in norm_data or 'female' in norm_data:
                            low_limit = norm_data[gender_key]['low']
                            high_limit = norm_data[gender_key]['high']
                        else:
                            low_limit = norm_data['low']
                            high_limit = norm_data['high']
                        
                        if low_limit <= actual_lab_val <= high_limit:
                            if shap_val > 0:
                                corrected_val = -abs(shap_val)

        corrected_features[feature] = corrected_val

    return corrected_features

In [57]:
for p in patients_json:
    sid = p["subject_id"]
    hid = p["hadm_id"]
    json_context = p['json_context']
    diagnoses_block = p.get('json_context', {}).get('diagnoses', {})

    diagnoses_icd = diagnoses_block.get('icd', [])
    diagnoses_ccsr = diagnoses_block.get('ccsr', [])
    lab_vals = list(json_context.get('laboratory_values', {}).keys())

    other_features = [
        'num_diagnoses',
        'num_chronic',
        'comorbidity_score',
        'num_medications_daily',
        'prior_admissions_12mo',
        'cumulative_procedures',
        'cumulative_medications',
        'num_procedures_daily',
        'gender_male',
        'age'
    ]

    allowed_features = set(diagnoses_icd + diagnoses_ccsr + lab_vals + other_features)

    delta = deltas[(deltas["subject_id"] == sid) & (deltas["hadm_id"] == hid)]
    delta_top = delta[delta['feature'].isin(allowed_features)].sort_values(by='delta_mean_abs', ascending=False).head(10)
    
    lime = limes[(limes["subject_id"] == sid) & (limes["hadm_id"] == hid)]
    lime_top = lime[lime['feature'].isin(allowed_features)].sort_values(by='lime_mean_abs', ascending=False).head(10)

    shap_cur = shap_combined[(shap_combined["hadm_id"] == hid)]
    shap_top = shap_cur[shap_cur['feature'].isin(allowed_features)].sort_values(by='shap_mean_abs', ascending=False).head(10)
    
    shap_cur_bck = shap_combined_bck[(shap_combined_bck["hadm_id"] == hid)]
    shap_top_bck = shap_cur_bck[shap_cur_bck['feature'].isin(allowed_features)].sort_values(by='shap_mean_abs', ascending=False).head(10)

    shap_cur_bck_filtered = shap_top_bck.copy()
    corrected_dict = correct_shap_signs(shap_top_bck, json_context)
    shap_cur_bck_filtered['shap_mean'] = shap_cur_bck_filtered['feature'].map(corrected_dict)

    p['delta_values'] = delta_top.set_index('feature')['delta_mean'].to_dict()
    p['lime_values'] = lime_top.set_index('feature')['lime_mean'].to_dict()
    p['shap_values'] = shap_top.set_index('feature')['shap_mean'].to_dict()
    p['shap_background_values'] = shap_top_bck.set_index('feature')['shap_mean'].to_dict()
    p['shap_background_filtered_values'] = shap_cur_bck_filtered.set_index('feature')['shap_mean'].to_dict()

### Judge

In [105]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("API_KEY")
BASE_URL = os.getenv("BASE_URL")

In [106]:
system_prompt = """
You are an experienced hospital physician with expertise in internal medicine, critical care, and clinical risk assessment.

Your task is to evaluate explanations generated by five different explainable AI methods for predicting 30-day hospital readmission.

You are NOT evaluating the prediction itself.
You are ONLY evaluating the quality of the explanations.

You will receive:

1. A patient's medical record from the final day of hospitalization.
The record contains:
- demographics,
- diagnoses,
- laboratory values,
- comorbidities,
- medications,
- procedures,
- hospitalization history,
- and the readmission true outcome (whether the patient was readmitted within 30 days).
The true outcome is provided only for clinical context.

Do NOT use the true outcome when evaluating explanation quality.
Evaluate only whether the explanations are clinically plausible.

2. Five anonymous explanation methods:
Method A
Method B
Method C
Method D
Method E

For each method you will receive ONLY the TOP-10 most important features ranked by absolute importance.

Each feature contains:

- feature name
- explanation score

Interpretation of explanation scores:

• Positive score:
this feature value increases the predicted readmission risk.

• Negative score:
this feature value decreases the predicted readmission risk.

The feature values should always be interpreted using the patient's medical record above.

The absolute value of the score represents the importance of the feature.
Larger absolute values indicate stronger influence on the prediction.

The ranking is already sorted by absolute importance.

Do NOT compare explanation magnitudes across methods because each method uses a different scale.
Use the scores only to determine:
- direction (positive vs negative),
- within-method ranking.

Evaluate every method independently according to the patient's medical record and current medical knowledge.

Evaluate the following criteria.

1. Clinical relevance (1-5)

Are the selected features clinically meaningful predictors of 30-day readmission?

1 = irrelevant
5 = highly clinically relevant

2. Direction correctness (1-5)

Is the sign of each feature medically reasonable?

1 = unrealistic directions
5 = very reasonable directions

3. Importance ranking (1-5)

Does the ordering of the TOP-10 features reflect their expected clinical importance?

1 = unrealistic ranking
5 = very realistic ranking

4. Completeness (1-5)

Does the explanation include the most important clinical factors while avoiding unimportant ones?

1 = only unimportant factors included
5 = most important factors included

5. Hallucination score (0-5)

Count how much the explanation appears to rely on implausible or medically unsupported factors.

0 = no hallucinations
5 = severe hallucinations

6. Overall explanation quality (1-5)

Overall usefulness and trustworthiness for a clinician.

1 = useless and not trustworthy
5 = very useful and trustworthy

Finally:

Select the single best explanation method.

Return ONLY valid JSON.

Return exactly the following schema:

{
  "best_method": "A",

  "methods": {

    "A": {
      "clinical_relevance": 5,
      "direction_correctness": 5,
      "importance_ranking": 5,
      "completeness": 4,
      "hallucination_score": 0,
      "overall_quality": 5,
      "reason": "Maximum 2 sentences."
    },

    "B": {
      "clinical_relevance": 4,
      "direction_correctness": 4,
      "importance_ranking": 4,
      "completeness": 4,
      "hallucination_score": 1,
      "overall_quality": 4,
      "reason": "Maximum 2 sentences."
    },

    "C": {
      "clinical_relevance": 3,
      "direction_correctness": 4,
      "importance_ranking": 3,
      "completeness": 3,
      "hallucination_score": 2,
      "overall_quality": 3,
      "reason": "Maximum 2 sentences."
    },

    "D": {
      "clinical_relevance": 5,
      "direction_correctness": 5,
      "importance_ranking": 4,
      "completeness": 5,
      "hallucination_score": 0,
      "overall_quality": 5,
      "reason": "Maximum 2 sentences."
    },

    "E": {
      "clinical_relevance": 1,
      "direction_correctness": 1,
      "importance_ranking": 3,
      "completeness": 2,
      "hallucination_score": 2,
      "overall_quality": 2,
      "reason": "Maximum 2 sentences."
    }

  }
}

Important requirements:

- Use ONLY the information provided.
- Do NOT infer unavailable information.
- Do NOT compare numerical scores across methods.
- Evaluate only clinical plausibility.
- Return valid JSON only.
- Do not use Markdown.
- Do not include any additional text before or after the JSON.
"""

In [107]:
import json
def build_prompt(patient):
    ehr = patient["json_context"]

    delta = patient['delta_values']
    lime = patient['lime_values']
    shap = patient['shap_values']
    shap_bck = patient['shap_background_values']
    shap_bck_filtered = patient['shap_background_filtered_values']

    return f"""
Patient's medical record of a hospitalization with information about 30-day readmission.

{json.dumps(ehr, indent=2)}

Below are explanations from five anonymous feature attribution methods:

Method A:
{json.dumps(delta, indent=2)}

Method B:
{json.dumps(lime, indent=2)}

Method C:
{json.dumps(shap, indent=2)}

Method D:
{json.dumps(shap_bck, indent=2)}

Method E:
{json.dumps(shap_bck_filtered, indent=2)}

Important:

The explanations contain ONLY the ten highest-ranked features.

The absence of a clinically important feature may reduce the completeness score.

The presence of an irrelevant feature may reduce the clinical relevance score.

A feature should be evaluated together with

• its value in the patient's record,

• its explanation sign,

• its rank within the method.
"""

In [108]:
from tqdm import tqdm

results = []
client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL 
)

for patient in tqdm(patients_json[:2]):
    print(patient)
    prompt = build_prompt(patient)

    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": build_prompt(patient)
        }
    ]
    print( build_prompt(patient))
    # response = client.chat.completions.create(
    #     model="gpt-oss-120b",
    #     messages=messages
    # )
    # print(response.choices[0].message.content)

    # results.append({
    #     "subject_id": patient["subject_id"],
    #     "hadm_id": patient["hadm_id"],
    #     "analysis": response
    # })

100%|██████████| 2/2 [00:00<00:00, 3914.42it/s]

{'subject_id': 14849966, 'hadm_id': 26616701, 'json_context': {'readmission': 'True', 'demographics': {'gender': 'Female', 'age': 78, 'length_of_stay': 19, 'race': 'White - Other European', 'insurance': 'Medicare', 'admission_type': 'Ew Emer.', 'discharge_location': 'Skilled Nursing Facility', 'admission_location': 'Transfer From Hospital'}, 'diagnoses': {'icd': ['Hyperlipidemia, unspecified (E785)', 'Gastroesophageal reflux disease without esophagitis (K219)', 'Atherosclerotic heart disease of native coronary artery without angina pectoris (I2510)', 'Acute kidney failure, unspecified (N179)', 'Unspecified atrial fibrillation (I4891)'], 'ccsr': ['Personal/family history of disease (FAC021)', 'Other specified status (FAC025)', 'Coronary atherosclerosis and other heart disease (CIR011)', 'Disorders of lipid metabolism (END010)', 'Diabetes mellitus with complication (END003)', 'Esophageal disorders (DIG004)', 'Cardiac dysrhythmias (CIR017)', 'Hypertension with complications and secondary 

In [109]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from openai import OpenAI
import json

client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)

def evaluate_patient(patient):

    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": build_prompt(patient)
        }
    ]

    try:
        response = client.chat.completions.create(
            model="gpt-oss-120b",
            messages=messages,
            temperature=0
        )

        text = response.choices[0].message.content
        if text.startswith("```"):
            text = text.replace("```json", "")
            text = text.replace("```", "")
            text = text.strip()

        try:
            parsed = json.loads(text)
        except Exception:
            parsed = None

        return {
            "subject_id": int(patient["subject_id"]),
            "hadm_id": int(patient["hadm_id"]),
            "response_text": text,
            "response_json": parsed,
            "success": parsed is not None
        }

    except Exception as e:

        return {
            "subject_id": int(patient["subject_id"]),
            "hadm_id": int(patient["hadm_id"]),
            "response_text": None,
            "response_json": None,
            "success": False,
            "error": str(e)
        }


results = []

with ThreadPoolExecutor(max_workers=8) as executor:

    futures = [
        executor.submit(evaluate_patient, patient)
        for patient in patients_json
    ]

    for future in tqdm(
        as_completed(futures),
        total=len(futures)
    ):

        result = future.result()
        with open("judge_results_with_filtered_shap.jsonl", "a") as f:
            f.write(json.dumps(result) + "\n")
            f.flush()

100%|██████████| 1440/1440 [2:09:25<00:00,  5.39s/it] 


In [4]:
import pandas as pd

results = pd.read_json(
    "../judge_results_with_filtered_shap.jsonl",
    lines=True
)

In [5]:
results['success'].value_counts()

success
True    1440
Name: count, dtype: int64

In [11]:
rows = []

for _, r in results.iterrows():

    methods = r["response_json"]["methods"]

    row = {
        "subject_id": r["subject_id"],
        "hadm_id": r["hadm_id"],
        "best_method": r["response_json"]["best_method"],
    }

    for method in ["A", "B", "C", "D", "E"]:
        for metric, value in methods[method].items():
            #if metric != "reason":
            row[f"{method}_{metric}"] = value

    rows.append(row)

judge_df = pd.DataFrame(rows)
judge_df

,subject_id,hadm_id,best_method,A_clinical_relevance,A_direction_correctness,A_importance_ranking,A_completeness,A_hallucination_score,A_overall_quality,A_reason,...,D_hallucination_score,D_overall_quality,D_reason,E_clinical_relevance,E_direction_correctness,E_importance_ranking,E_completeness,E_hallucination_score,E_overall_quality,E_reason
0,18375223,24980184,D,2,2,2,2,2,2,"Only a few truly relevant risk factors appear,...",...,0,4,Captures the most important risk drivers with ...,4,2,4,4,0,3,Contains the right set of predictors but sever...
1,10184327,21418533,E,2,2,2,2,1,2,Features are mostly relevant but many have imp...,...,1,3,"Captures many key risk factors (age, comorbidi...",5,4,4,5,0,5,Provides a comprehensive set of established re...
2,17940493,28973861,D,2,1,2,2,1,1,Key predictors are present but most signs are ...,...,0,5,Features align with major readmission risk dri...,5,4,4,5,0,5,"Same as D, providing a comprehensive and plaus..."
3,17154976,23809945,E,2,2,2,2,2,2,Includes some relevant variables but many have...,...,1,4,Captures most major risk factors and orders th...,4,4,4,4,1,5,Provides a comprehensive set of clinically imp...
4,15146009,21716326,E,3,2,3,3,2,2,Includes prior admissions and CAD but many lab...,...,1,4,Strong on admissions and disease burden; some ...,4,4,4,4,1,4,"Correctly flags age, diabetes, and WBC as risk..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1435,12867386,25322150,A,4,3,4,4,0,4,Includes the most relevant risk factors and pl...,...,0,3,Highlights medication burden and age positivel...,2,1,2,3,0,2,All major risk factors are given negative cont...
1436,16900315,27971624,A,3,3,3,3,0,3,Includes the most pertinent risk factors (medi...,...,0,1,Places several irrelevant features first and a...,1,1,2,1,0,1,Nearly all top features have signs opposite to...
1437,18447676,28733755,A,4,3,2,3,0,3,Features are generally relevant and include ke...,...,0,3,"Includes appropriate risk factors, yet several...",3,1,3,3,0,2,Features are relevant but most signs are rever...
1438,19090467,21236644,D,2,2,2,2,0,2,Features are largely zero-valued or irrelevant...,...,0,4,"Ranks age first, a strong predictor, and inclu...",3,1,3,3,0,2,Features mirror D but most direction signs are...


In [24]:
judge_df = judge_df.merge(

    hospital_df[
        [
            "hadm_id",
            "group",
            "context_type",
            "true_class",
            "predicted_class",
            "predicted_proba",
            "age"
        ]
    ],

    on="hadm_id",
    how="left"
)

In [27]:
judge_df.head(2)

,subject_id,hadm_id,best_method,A_clinical_relevance,A_direction_correctness,A_importance_ranking,A_completeness,A_hallucination_score,A_overall_quality,A_reason,...,E_completeness,E_hallucination_score,E_overall_quality,E_reason,group,context_type,true_class,predicted_class,predicted_proba,age
0,18375223,24980184,D,2,2,2,2,2,2,"Only a few truly relevant risk factors appear,...",...,4,0,3,Contains the right set of predictors but sever...,TP,full,1,1,0.216169,85
1,10184327,21418533,E,2,2,2,2,1,2,Features are mostly relevant but many have imp...,...,5,0,5,Provides a comprehensive set of established re...,TP,full,1,1,0.215882,82


### Analysis

In [28]:
metrics = [
    "clinical_relevance",
    "direction_correctness",
    "importance_ranking",
    "completeness",
    "hallucination_score",
    "overall_quality"
]

methods = ["A", "B", "C", "D", "E"]


def summarize_subset(df, group, context):

    if len(df) == 0:
        return None

    row = {
        "group": group,
        "context": context,
        "N": len(df)
    }

    for method in methods:

        for metric in metrics:

            values = df[f"{method}_{metric}"]

            row[f"{method}_{metric}_mean"] = values.mean()
            row[f"{method}_{metric}_std"] = values.std()

    return row

#### Delta

In [29]:
method = "A"

results = []
groups = ["TP", "TN", "FP", "FN"]
for g in groups:

    df_group = judge_df[judge_df.group == g]

    for c in contexts:

        df = df_group[df_group.context_type == c]

        if len(df) == 0:
            continue

        row = {
            "group": g,
            "context": c,
            "N": len(df)
        }

        for metric in metrics:

            row[metric] = (
                f"{df[f'{method}_{metric}'].mean():.2f} ± "
                f"{df[f'{method}_{metric}'].std():.2f}"
            )

        row["wins"] = (
            (df.best_method == method).mean()
        )

        results.append(row)

summary_A = pd.DataFrame(results)
summary_A

,group,context,N,clinical_relevance,direction_correctness,importance_ranking,completeness,hallucination_score,overall_quality,wins
0,TP,full,90,3.26 ± 0.83,2.13 ± 0.48,2.69 ± 0.73,2.63 ± 0.59,0.78 ± 0.87,2.40 ± 0.68,0.022222
1,TP,diagnoses_only,90,3.43 ± 0.78,2.44 ± 0.91,2.67 ± 0.73,2.94 ± 0.71,0.42 ± 0.64,2.58 ± 0.83,0.077778
2,TP,labs_only,90,3.32 ± 0.93,2.38 ± 0.77,2.72 ± 0.79,2.86 ± 0.83,0.50 ± 0.80,2.66 ± 0.86,0.088889
3,TP,empty,90,3.58 ± 0.92,2.36 ± 0.88,2.72 ± 0.91,3.13 ± 0.81,0.36 ± 0.80,2.63 ± 0.94,0.211111
4,TN,full,90,2.87 ± 0.97,1.97 ± 0.69,2.32 ± 0.82,2.30 ± 0.66,0.73 ± 0.93,2.11 ± 0.74,0.033333
5,TN,diagnoses_only,90,3.31 ± 0.74,2.01 ± 0.49,2.63 ± 0.74,2.96 ± 0.75,0.24 ± 0.53,2.31 ± 0.57,0.044444
6,TN,labs_only,90,2.94 ± 0.96,1.94 ± 0.78,2.34 ± 0.88,2.47 ± 0.71,0.50 ± 0.84,2.12 ± 0.76,0.066667
7,TN,empty,90,3.90 ± 0.81,2.03 ± 0.80,2.89 ± 0.88,3.48 ± 0.75,0.07 ± 0.44,2.57 ± 0.75,0.177778
8,FP,full,90,3.48 ± 0.84,2.41 ± 0.81,2.89 ± 0.79,2.83 ± 0.72,0.49 ± 0.78,2.73 ± 0.85,0.066667
9,FP,diagnoses_only,90,3.46 ± 0.78,2.14 ± 0.70,2.68 ± 0.78,3.02 ± 0.72,0.42 ± 0.87,2.46 ± 0.72,0.033333


#### LIME

In [30]:
method = "B"

results = []
groups = ["TP", "TN", "FP", "FN"]
for g in groups:

    df_group = judge_df[judge_df.group == g]

    for c in contexts:

        df = df_group[df_group.context_type == c]

        if len(df) == 0:
            continue

        row = {
            "group": g,
            "context": c,
            "N": len(df)
        }

        for metric in metrics:

            row[metric] = (
                f"{df[f'{method}_{metric}'].mean():.2f} ± "
                f"{df[f'{method}_{metric}'].std():.2f}"
            )

        row["wins"] = (
            (df.best_method == method).mean()
        )

        results.append(row)

summary_B = pd.DataFrame(results)
summary_B

,group,context,N,clinical_relevance,direction_correctness,importance_ranking,completeness,hallucination_score,overall_quality,wins
0,TP,full,90,2.84 ± 0.63,1.94 ± 0.59,2.36 ± 0.66,2.36 ± 0.57,1.08 ± 0.96,2.13 ± 0.69,0.055556
1,TP,diagnoses_only,90,3.30 ± 0.66,2.17 ± 0.60,2.66 ± 0.67,3.07 ± 0.65,0.51 ± 0.81,2.48 ± 0.69,0.055556
2,TP,labs_only,90,2.74 ± 0.66,1.96 ± 0.60,2.46 ± 0.67,2.39 ± 0.61,1.07 ± 1.10,2.18 ± 0.61,0.011111
3,TP,empty,90,3.31 ± 0.91,2.18 ± 0.82,2.71 ± 0.90,3.20 ± 0.85,0.46 ± 0.80,2.43 ± 0.91,0.200000
4,TN,full,90,2.99 ± 0.69,2.18 ± 0.59,2.70 ± 0.71,2.61 ± 0.71,0.79 ± 0.98,2.50 ± 0.71,0.122222
5,TN,diagnoses_only,90,3.73 ± 0.68,2.48 ± 0.72,3.30 ± 0.79,3.59 ± 0.73,0.16 ± 0.45,3.01 ± 0.84,0.333333
6,TN,labs_only,90,3.31 ± 0.79,2.44 ± 0.72,3.02 ± 0.87,2.93 ± 0.85,0.49 ± 0.82,2.86 ± 0.93,0.344444
7,TN,empty,90,4.22 ± 0.63,2.68 ± 0.75,3.43 ± 0.72,4.00 ± 0.56,0.03 ± 0.18,3.21 ± 0.76,0.477778
8,FP,full,90,2.74 ± 0.61,1.94 ± 0.61,2.29 ± 0.64,2.20 ± 0.58,0.92 ± 1.05,2.12 ± 0.58,0.022222
9,FP,diagnoses_only,90,3.32 ± 0.58,2.14 ± 0.66,2.76 ± 0.69,3.18 ± 0.70,0.39 ± 0.73,2.57 ± 0.65,0.055556


#### SHAP

In [31]:
method = "C"

results = []
groups = ["TP", "TN", "FP", "FN"]
for g in groups:

    df_group = judge_df[judge_df.group == g]

    for c in contexts:

        df = df_group[df_group.context_type == c]

        if len(df) == 0:
            continue

        row = {
            "group": g,
            "context": c,
            "N": len(df)
        }

        for metric in metrics:

            row[metric] = (
                f"{df[f'{method}_{metric}'].mean():.2f} ± "
                f"{df[f'{method}_{metric}'].std():.2f}"
            )

        row["wins"] = (
            (df.best_method == method).mean()
        )

        results.append(row)

summary_C = pd.DataFrame(results)
summary_C

,group,context,N,clinical_relevance,direction_correctness,importance_ranking,completeness,hallucination_score,overall_quality,wins
0,TP,full,90,3.77 ± 0.69,2.77 ± 0.69,3.46 ± 0.78,3.46 ± 0.67,0.53 ± 0.71,3.28 ± 0.84,0.166667
1,TP,diagnoses_only,90,3.51 ± 0.74,2.27 ± 0.68,3.03 ± 0.77,3.40 ± 0.79,0.44 ± 0.78,2.74 ± 0.74,0.055556
2,TP,labs_only,90,3.72 ± 0.76,2.79 ± 0.85,3.34 ± 0.86,3.43 ± 0.74,0.52 ± 0.78,3.22 ± 0.92,0.200000
3,TP,empty,90,3.46 ± 0.74,2.24 ± 0.75,2.77 ± 0.78,3.38 ± 0.77,0.54 ± 0.95,2.56 ± 0.72,0.022222
4,TN,full,90,3.27 ± 0.73,2.23 ± 0.85,2.86 ± 0.79,3.07 ± 0.76,0.63 ± 0.89,2.67 ± 0.87,0.055556
5,TN,diagnoses_only,90,3.42 ± 0.70,1.84 ± 0.58,2.96 ± 0.86,3.54 ± 0.80,0.27 ± 0.67,2.40 ± 0.60,0.011111
6,TN,labs_only,90,3.46 ± 0.84,2.11 ± 0.89,2.97 ± 0.91,3.36 ± 0.77,0.37 ± 0.76,2.61 ± 0.82,0.055556
7,TN,empty,90,4.00 ± 0.85,1.92 ± 0.75,3.32 ± 0.87,4.02 ± 0.79,0.09 ± 0.39,2.58 ± 0.75,0.033333
8,FP,full,90,3.87 ± 0.67,2.94 ± 0.81,3.40 ± 0.76,3.54 ± 0.77,0.43 ± 0.69,3.39 ± 0.79,0.155556
9,FP,diagnoses_only,90,3.49 ± 0.64,2.23 ± 0.72,2.90 ± 0.79,3.40 ± 0.67,0.41 ± 0.83,2.64 ± 0.72,0.066667


#### SHAP + background

In [32]:
method = "D"

results = []
groups = ["TP", "TN", "FP", "FN"]
for g in groups:

    df_group = judge_df[judge_df.group == g]

    for c in contexts:

        df = df_group[df_group.context_type == c]

        if len(df) == 0:
            continue

        row = {
            "group": g,
            "context": c,
            "N": len(df)
        }

        for metric in metrics:

            row[metric] = (
                f"{df[f'{method}_{metric}'].mean():.2f} ± "
                f"{df[f'{method}_{metric}'].std():.2f}"
            )

        row["wins"] = (
            (df.best_method == method).mean()
        )

        results.append(row)

summary_D = pd.DataFrame(results)
summary_D

,group,context,N,clinical_relevance,direction_correctness,importance_ranking,completeness,hallucination_score,overall_quality,wins
0,TP,full,90,4.19 ± 0.60,2.90 ± 0.69,3.68 ± 0.73,3.93 ± 0.67,0.46 ± 0.69,3.63 ± 0.76,0.266667
1,TP,diagnoses_only,90,4.03 ± 0.74,2.72 ± 0.84,3.49 ± 0.85,3.88 ± 0.79,0.26 ± 0.59,3.46 ± 0.94,0.388889
2,TP,labs_only,90,4.27 ± 0.67,3.02 ± 0.75,3.83 ± 0.67,4.08 ± 0.74,0.34 ± 0.60,3.78 ± 0.78,0.488889
3,TP,empty,90,3.99 ± 0.76,2.78 ± 0.87,3.37 ± 0.93,3.97 ± 0.85,0.32 ± 0.72,3.39 ± 0.94,0.455556
4,TN,full,90,3.82 ± 0.63,2.69 ± 0.74,3.46 ± 0.74,3.62 ± 0.65,0.32 ± 0.58,3.28 ± 0.86,0.155556
5,TN,diagnoses_only,90,3.74 ± 0.73,2.22 ± 0.70,3.29 ± 0.80,3.81 ± 0.75,0.18 ± 0.51,2.83 ± 0.77,0.122222
6,TN,labs_only,90,3.87 ± 0.84,2.59 ± 0.81,3.37 ± 0.83,3.84 ± 0.83,0.23 ± 0.58,3.22 ± 0.93,0.333333
7,TN,empty,90,4.20 ± 0.75,2.22 ± 0.82,3.51 ± 0.82,4.27 ± 0.73,0.06 ± 0.23,2.94 ± 0.94,0.211111
8,FP,full,90,4.16 ± 0.75,2.94 ± 0.84,3.66 ± 0.82,3.92 ± 0.82,0.40 ± 0.73,3.63 ± 0.94,0.388889
9,FP,diagnoses_only,90,3.94 ± 0.72,2.60 ± 0.86,3.33 ± 0.75,3.91 ± 0.76,0.27 ± 0.65,3.23 ± 0.97,0.255556


#### SHAP + background filtered

In [33]:
method = "E"

results = []
groups = ["TP", "TN", "FP", "FN"]
for g in groups:

    df_group = judge_df[judge_df.group == g]

    for c in contexts:

        df = df_group[df_group.context_type == c]

        if len(df) == 0:
            continue

        row = {
            "group": g,
            "context": c,
            "N": len(df)
        }

        for metric in metrics:

            row[metric] = (
                f"{df[f'{method}_{metric}'].mean():.2f} ± "
                f"{df[f'{method}_{metric}'].std():.2f}"
            )

        row["wins"] = (
            (df.best_method == method).mean()
        )

        results.append(row)

summary_E = pd.DataFrame(results)
summary_E

,group,context,N,clinical_relevance,direction_correctness,importance_ranking,completeness,hallucination_score,overall_quality,wins
0,TP,full,90,4.31 ± 0.70,3.22 ± 0.93,3.86 ± 0.71,4.09 ± 0.76,0.39 ± 0.71,3.89 ± 0.95,0.488889
1,TP,diagnoses_only,90,3.89 ± 1.01,2.93 ± 1.26,3.47 ± 0.88,3.86 ± 0.87,0.22 ± 0.56,3.31 ± 1.24,0.422222
2,TP,labs_only,90,3.86 ± 0.86,2.56 ± 1.06,3.56 ± 0.82,3.98 ± 0.79,0.52 ± 0.86,3.13 ± 0.95,0.211111
3,TP,empty,90,3.02 ± 1.06,1.74 ± 1.10,2.71 ± 0.81,3.26 ± 1.03,0.60 ± 1.12,2.14 ± 1.06,0.111111
4,TN,full,90,4.03 ± 0.64,3.47 ± 0.96,3.71 ± 0.69,3.82 ± 0.65,0.32 ± 0.60,3.84 ± 0.87,0.633333
5,TN,diagnoses_only,90,4.10 ± 0.75,3.21 ± 1.10,3.59 ± 0.86,4.13 ± 0.67,0.13 ± 0.45,3.58 ± 0.99,0.488889
6,TN,labs_only,90,3.78 ± 0.79,2.66 ± 0.88,3.32 ± 0.78,3.78 ± 0.83,0.29 ± 0.64,3.08 ± 0.89,0.200000
7,TN,empty,90,3.99 ± 0.84,2.07 ± 1.01,3.30 ± 0.74,4.07 ± 0.80,0.04 ± 0.26,2.57 ± 0.94,0.100000
8,FP,full,90,4.13 ± 0.75,3.16 ± 1.06,3.68 ± 0.78,3.96 ± 0.76,0.37 ± 0.71,3.59 ± 0.95,0.366667
9,FP,diagnoses_only,90,4.24 ± 0.85,3.34 ± 1.30,3.80 ± 0.94,4.24 ± 0.72,0.30 ± 0.71,3.78 ± 1.21,0.588889


#### Overall

In [34]:
metrics_without_hall = [
    "clinical_relevance",
    "direction_correctness",
    "importance_ranking",
    "completeness",
    "overall_quality"
]

for method in ["A", "B", "C", "D", "E"]:

    judge_df[f"{method}_total_score"] = (
        judge_df[[f"{method}_{m}" for m in metrics_without_hall]]
        .sum(axis=1)
    )

In [35]:
summary = {}

for method in ["A", "B", "C", "D", "E"]:

    row = {}

    for metric in metrics:
        row[metric] = (
            f"{judge_df[f'{method}_{metric}'].mean():.2f} ± "
            f"{judge_df[f'{method}_{metric}'].std():.2f}"
        )

    row["total_score"] = (
        f"{judge_df[f'{method}_total_score'].mean():.2f} ± "
        f"{judge_df[f'{method}_total_score'].std():.2f}"
    )

    row["wins"] = (
        f"{100 * (judge_df.best_method == method).mean():.1f}%"
    )

    summary[method] = row

summary = pd.DataFrame(summary).T
summary

,clinical_relevance,direction_correctness,importance_ranking,completeness,hallucination_score,overall_quality,total_score,wins
A,3.31 ± 0.91,2.15 ± 0.75,2.59 ± 0.82,2.83 ± 0.79,0.44 ± 0.78,2.41 ± 0.79,13.28 ± 3.45,8.4%
B,3.30 ± 0.82,2.28 ± 0.76,2.82 ± 0.83,2.99 ± 0.88,0.56 ± 0.86,2.62 ± 0.86,14.01 ± 3.63,20.4%
C,3.54 ± 0.77,2.27 ± 0.82,3.01 ± 0.86,3.41 ± 0.78,0.44 ± 0.79,2.72 ± 0.83,14.95 ± 3.32,6.7%
D,3.98 ± 0.75,2.63 ± 0.83,3.44 ± 0.84,3.88 ± 0.78,0.28 ± 0.62,3.29 ± 0.95,17.21 ± 3.54,30.2%
E,3.86 ± 0.92,2.78 ± 1.19,3.42 ± 0.86,3.85 ± 0.85,0.31 ± 0.70,3.21 ± 1.15,17.13 ± 4.24,34.3%


### SHAP + background vs SHAP + background filtered

In [36]:
metrics = [
    "clinical_relevance",
    "direction_correctness",
    "importance_ranking",
    "completeness",
    "overall_quality"
]

for method in ["D", "E"]:
    judge_df[f"{method}_total"] = (
        judge_df[[f"{method}_{m}" for m in metrics]].sum(axis=1)
        - judge_df[f"{method}_hallucination_score"]
    )

In [61]:
import json
judge_df["diff_total"] = judge_df["E_total"] - judge_df["D_total"]

judge_df = judge_df.sort_values("diff_total", ascending=False)[[
    "subject_id",
    "hadm_id",
    "diff_total",
    "D_total",
    "E_total",
    "best_method",
    "D_reason",
    "E_reason"
]]

patients_lookup = {
    (p["subject_id"], p["hadm_id"]): p
    for p in patients_json
}

cols = [
    "subject_id",
    "hadm_id",
    "diff_total",
    "D_total",
    "E_total",
    "best_method",
    "D_reason",
    "E_reason"
]

print("=" * 100)
print("TOP 5: SHAP+rules better SHAP+background")
print("=" * 100)

for _, row in judge_df.sort_values("diff_total", ascending=False).head(5).iterrows():

    print(f"\nPatient: {row.subject_id}   HADM: {row.hadm_id}")
    patient = patients_lookup[(row.subject_id, row.hadm_id)]

    print("\nPATIENT")
    print(json.dumps(patient["json_context"], indent=2))

    print("\nSHAP + background")
    print(json.dumps(patient["shap_background_values"], indent=2))

    print("\nSHAP + rules")
    print(json.dumps(patient["shap_background_filtered_values"], indent=2))
    
    print(f"Difference: {row.diff_total:+.1f}")
    print(f"D total: {row.D_total:.1f}")
    print(f"E total: {row.E_total:.1f}")
    print(f"Winner: {row.best_method}")

    print("\nD:")
    print(row.D_reason)

    print("\nE:")
    print(row.E_reason)

    print("-" * 100)


print("\n\n")
print("=" * 100)
print("TOP 5: SHAP+rules worse SHAP+background")
print("=" * 100)

for _, row in judge_df.sort_values("diff_total").head(5).iterrows():

    print(f"\nPatient: {row.subject_id}   HADM: {row.hadm_id}")
    patient = patients_lookup[(row.subject_id, row.hadm_id)]

    print("\nPATIENT")
    print(json.dumps(patient["json_context"], indent=2))

    print("\nSHAP + background")
    print(json.dumps(patient["shap_background_values"], indent=2))

    print("\nSHAP + rules")
    print(json.dumps(patient["shap_background_filtered_values"], indent=2))
          
    print(f"Difference: {row.diff_total:+.1f}")
    print(f"D total: {row.D_total:.1f}")
    print(f"E total: {row.E_total:.1f}")
    print(f"Winner: {row.best_method}")

    print("\nD:")
    print(row.D_reason)

    print("\nE:")
    print(row.E_reason)

    print("-" * 100)

TOP 5: SHAP+rules better SHAP+background

Patient: 12240183   HADM: 20766526

PATIENT
{
  "readmission": "True",
  "demographics": {
    "gender": "Male",
    "age": 23,
    "length_of_stay": 2,
    "race": "Hispanic Or Latino",
    "insurance": "Medicaid",
    "admission_type": "Eu Observation",
    "discharge_location": "Unknown",
    "admission_location": "Emergency Room"
  },
  "diagnoses": {
    "icd": [],
    "ccsr": []
  },
  "laboratory_values": {},
  "clinical_indicators": {
    "num_diagnoses": 2.0,
    "num_chronic": 0.0,
    "comorbidity_score": 0.0,
    "num_medications_daily": 0.0,
    "prior_admissions_12mo": 1.0,
    "cumulative_procedures": 0.0,
    "cumulative_medications": 0.0,
    "num_procedures_daily": 0.0
  }
}

SHAP + background
{
  "age": 0.3155300267027291,
  "cumulative_medications": 0.2616261199719156,
  "num_medications_daily": 0.2329014103940198,
  "num_chronic": -0.1004618135985492,
  "num_diagnoses": -0.0978292717940848,
  "comorbidity_score": -0.0556570

In [40]:
metrics = [
    "clinical_relevance",
    "direction_correctness",
    "importance_ranking",
    "completeness",
    "hallucination_score",
    "overall_quality"
]

comparison = pd.DataFrame({
    metric: (
        judge_df[f"E_{metric}"] - judge_df[f"D_{metric}"]
    ).value_counts().sort_index()
    for metric in metrics
}).fillna(0)

comparison

,clinical_relevance,direction_correctness,importance_ranking,completeness,hallucination_score,overall_quality
-4,1.0,3,0.0,1.0,0.0,12
-3,30.0,56,8.0,12.0,5.0,79
-2,106.0,154,79.0,46.0,7.0,149
-1,243.0,289,239.0,191.0,89.0,334
0,738.0,265,806.0,941.0,1226.0,288
1,261.0,418,230.0,208.0,69.0,392
2,55.0,196,73.0,37.0,38.0,143
3,6.0,56,5.0,4.0,6.0,40
4,0.0,3,0.0,0.0,0.0,3
